# **Day 76: Working with Web APIs**

## **What is an API?**
An **API** (Application Programming Interface) is a way for one program to **ask another program for data** over the internet.

**Real-life example:** In a restaurant, you (the customer) do not go into the kitchen. You tell the **waiter** what you want, and the waiter brings the food. The API is the **waiter** between our program and the server's data.

A **web API** is simply a URL that returns **data** (usually JSON) instead of a web page.

We use the `requests` library (`pip install requests`). For practice, we use **Open-Meteo**, a free weather API that does not need any key or login.

In [1]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 22.57,
    "longitude": 88.36,
    "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": "Asia/Kolkata",
}
r = requests.get(url, params=params, timeout=10)

print("Status code:", r.status_code)
print("Full URL:", r.url)
print("Content type:", r.headers["Content-Type"])

Status code: 200
Full URL: https://api.open-meteo.com/v1/forecast?latitude=22.57&longitude=88.36&current=temperature_2m%2Crelative_humidity_2m%2Cwind_speed_10m&timezone=Asia%2FKolkata
Content type: application/json; charset=utf-8


In [2]:
data = r.json()                         # Convert the JSON text into a Python dictionary
current = data["current"]
print(f"Kolkata right now: {current['temperature_2m']} C, humidity {current['relative_humidity_2m']}%, "
      f"wind {current['wind_speed_10m']} km/h")
print("Measured at:", current["time"])

Kolkata right now: 27.0 C, humidity 84%, wind 3.1 km/h
Measured at: 2026-09-27T07:30


(These are **live values** from the moment this notebook was run.)

## **HTTP Basics**
**Methods (what we want to do):**
| Method | Meaning |
|---|---|
| GET | Read data |
| POST | Send new data |
| PUT / PATCH | Change data |
| DELETE | Remove data |

**Status codes (what the server answers):**
| Code | Meaning |
|---|---|
| 200 | OK |
| 201 | Created |
| 400 | Bad request (we sent something wrong) |
| 401 / 403 | Not allowed |
| 404 | Not found |
| 429 | Too many requests (slow down!) |
| 500 | Server error |

## **A 7-Day Forecast for Several Places**

In [3]:
places = {
    "Kolkata": (22.57, 88.36),
    "Siliguri": (26.73, 88.40),
    "Darjeeling": (27.04, 88.27),
    "Midnapore": (22.43, 87.32),
    "Digha": (21.63, 87.51),
}

def forecast(lat, lon, days=7):
    r = requests.get(url, params={
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "Asia/Kolkata", "forecast_days": days,
    }, timeout=10)
    r.raise_for_status()                 # Raise an error for 4xx / 5xx codes
    return r.json()["daily"]

for name, (lat, lon) in places.items():
    try:
        d = forecast(lat, lon)
        rainy_days = sum(1 for p in d["precipitation_sum"] if p and p > 1)
        print(f"{name:<11} max {max(d['temperature_2m_max']):>5} C   min {min(d['temperature_2m_min']):>5} C   "
              f"rainy days: {rainy_days}/7")
    except requests.RequestException as e:
        print(name, "failed:", e)

Kolkata     max  32.4 C   min  25.7 C   rainy days: 3/7


Siliguri    max  31.8 C   min  23.1 C   rainy days: 3/7


Darjeeling  max  22.2 C   min  13.1 C   rainy days: 5/7


Midnapore   max  32.6 C   min  24.0 C   rainy days: 2/7


Digha       max  31.9 C   min  24.6 C   rainy days: 4/7


## **Example from GIS: Elevation Profile Along a Route**
An **elevation profile** shows how the height of the land changes along a route. It is used for planning roads, railways and trekking routes.

Let us get the profile of the famous **Darjeeling Himalayan Railway** route from Siliguri to Darjeeling. The Open-Meteo elevation API accepts **many points in one request**.

(The route points are approximate locations of stations.)

In [4]:
route = [
    ("Siliguri", 26.7271, 88.3953),
    ("Sukna", 26.7880, 88.3470),
    ("Rongtong", 26.8200, 88.3200),
    ("Tindharia", 26.8540, 88.3350),
    ("Kurseong", 26.8800, 88.2780),
    ("Sonada", 26.9580, 88.2700),
    ("Ghum", 27.0070, 88.2530),
    ("Darjeeling", 27.0410, 88.2663),
]

r = requests.get("https://api.open-meteo.com/v1/elevation", params={
    "latitude": ",".join(str(p[1]) for p in route),
    "longitude": ",".join(str(p[2]) for p in route),
}, timeout=15)
heights = r.json()["elevation"]

print(f"{'Station':<11}{'Height':>8}")
for (name, lat, lon), h in zip(route, heights):
    print(f"{name:<11}{h:>7.0f} m  {'#' * int(h // 100)}")

highest = max(zip(route, heights), key=lambda x: x[1])
print(f"\nThe route climbs about {heights[-1] - heights[0]:.0f} m. The highest point is {highest[0][0]} ({highest[1]:.0f} m).")

Station      Height
Siliguri       126 m  #
Sukna          183 m  #
Rongtong       320 m  ###
Tindharia      818 m  ########
Kurseong      1498 m  ##############
Sonada        1918 m  ###################
Ghum          2243 m  ######################
Darjeeling    2100 m  #####################

The route climbs about 1974 m. The highest point is Ghum (2243 m).


Let us understand the code above:
- We join all latitudes with commas (`"26.7271,26.788,..."`) so that **one request** gets all the heights.
- `zip(route, heights)` puts each station together with its height.
- The result shows that **Ghum** is the highest point on this railway, which is a well-known fact (Ghum is one of the highest railway stations in India).

## **Handling Errors**
Things that can go wrong: no internet, a timeout, a wrong request, or a server problem.

In [5]:
def safe_get(address, **kwargs):
    try:
        r = requests.get(address, timeout=5, **kwargs)
        r.raise_for_status()
        return r.json()
    except requests.Timeout:
        print("Timed out")
    except requests.HTTPError as e:
        print("HTTP error:", e.response.status_code)
    except requests.ConnectionError:
        print("Could not connect")

safe_get(url, params={"latitude": 999, "longitude": 88})              # Latitude 999 is invalid
safe_get("https://this-domain-surely-does-not-exist-12345.com")

HTTP error: 400
Could not connect


## **API Keys**
Many APIs need a secret **API key**. **Never write the key directly in code that you upload to GitHub.** Keep it in an environment variable instead:
```python
import os
API_KEY = os.environ["MY_API_KEY"]
```

## **Without Installing Anything: urllib**
Python's built-in `urllib` can also call APIs.

In [6]:
from urllib.request import urlopen
from urllib.parse import urlencode
import json

query = urlencode({"latitude": 26.73, "longitude": 88.40, "current": "temperature_2m"})
with urlopen(f"https://api.open-meteo.com/v1/forecast?{query}", timeout=10) as response:
    print("Siliguri now:", json.load(response)["current"]["temperature_2m"], "C")

Siliguri now: 26.7 C


In [7]:
# Quick question
# What does the status code 404 mean?